# Part 2: Write an LRU policy

In Part 1, you saw that agentic LLM calls share long prompt prefixes. An inference engine can retain KV-cache blocks after a call finishes so subsequent calls with matching prefixes can skip redundant prefill computation. However, retained cache blocks consume memory that active calls also need.

In this part, you will implement a least-recently-used (LRU) eviction policy and explore how cache eviction, concurrency caps, batch budgets, and dispatch order affect end-to-end job latency. Policies implement the `Policy` interface in `agentkv/policy.py`, which provides eight hooks (`target_concurrency` and `choose_victim` are required; the others are optional and default to FCFS dispatch/admission, least-loaded routing, and Vidur's vLLM V1 scheduler with a 2048-token mixed batch budget):

```python
class MyPolicy(Policy):
    def on_event(self, event, state): ...                              # observe events
    def target_concurrency(self, view, state) -> int: ...              # set the cap (1–16)
    def choose_victim(self, view, candidates, state) -> str: ...       # evict a cached block
    def dispatch_order(self, view, ready, state) -> list[str]: ...     # order ready calls
    def admission_order(self, view, waiting, state) -> list[str]: ...  # order engine admissions
    def route_call(self, view, call, state) -> int: ...                # select a replica
    def batch_token_budget(self, view, state) -> int: ...              # choose the next batch's budget
    def choose_preempted(self, view, running, state) -> str: ...       # preempt a running call
```

You will complete two policy files:

- `policies/reference.py` (Task 1): LRU eviction. This is the reference policy that every Part 3 speedup is measured against.
- `policies/prefix_policy.py` (Tasks 2 and 3): eviction that keeps prefix trees reusable, and routing that trades cache affinity against load balance across two GPU replicas.

See the [Part 2 handout](../docs/part2_handout.md) and `agentkv/policy.py` for the hook contracts.

![Policy hooks in the application and inference engine.](../docs/figures/system.svg)

| Section | Task | Estimated time |
|---|---|---|
| 1–2 | Cache states; implement LRU | 1–1.5 h |
| 3–4 | Concurrency caps on one and two replicas | 1 h |
| 5 | Implement prefix-aware eviction | 1.5–2 h |
| 6 | Implement cache-affinity routing; sweep its slack | 1.5 h |
| Questions | Answer Q1–Q3 | 2 h |

**Estimated time:** 7–8 hours.

**Grading (20%):** LRU 5%, prefix-aware eviction 4%, cache-affinity routing 3%, sweeps and answers to Q1–Q3 8%.

## Evaluator setup

Sections 1–5 simulate a single model replica (Llama-3-8B on one NVIDIA A100 80 GB GPU) with 16-token KV-cache blocks and Vidur execution-time predictions. Each replay fixes the workload, KV-cache quota, and batch limits, starting from an empty cache. (Sections 4 and 6 and all of Part 3 use two replicas; always compare policies under the same replica count. See [Evaluator configuration](../README.md#evaluator-configuration) for details.)

## Before you begin

Complete the [environment setup](../README.md#setup) before running this notebook, and select **CS 378 Assignment 4** as the notebook kernel.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt

from agentkv import viz
from agentkv.notebook import check, repo_root

os.chdir(repo_root())          # every path below is relative to the repository root
plt.rcParams["figure.dpi"] = 100

from agentkv import score
from agentkv.harness import BlockHarness
from agentkv.policy import Policy
from agentkv.sandbox.loader import instantiate, load_policy_class


def load_policy(path, **params):
    """Instantiate the policy class defined in a policy file."""
    return instantiate(load_policy_class(Path(path).read_text(), path), params)


def show(h, note=""):
    """Print the cache state: [A*] is active, [B] is cached, and [ ] is empty."""
    active, cached = h.active(), h.reusable()
    slots = [f"[{n}*]" for n in active] + [f"[{n}]" for n in cached] + ["[ ]"] * h.empty_slots()
    evicted = f"   evicted so far: {', '.join(h.evicted)}" if h.evicted else ""
    print(f"{' '.join(slots):24} {note}{evicted}")

## 1. KV-cache allocation and reuse

Within a replica, prompts that share a common prefix can reuse the full 16-token KV blocks for that prefix:

![Three prompts and the prefix blocks they share.](../docs/figures/kv_prefix.svg)

To trace block allocation and eviction step by step, we use `BlockHarness`, a small teaching cache with letter-labeled 1-token blocks:

- **Active (`[A*]`):** Held by at least one running call (`refs > 0`). Active blocks can never be evicted.
- **Cached (`[A]`):** Released by all calls (`refs == 0`), but retained in memory so a future call with the same prefix can reuse it (`hit`). Only cached blocks appear in `candidates` when `choose_victim` is called.
- **Empty (`[ ]`):** Unused slot. The engine always fills empty slots before calling `choose_victim`.

The cell below runs a 3-slot cache under FIFO eviction (`candidates` are passed oldest-released first, so `candidates[0]` is the block whose final release happened earliest). Step through the output to see why allocating `D` triggers an eviction while earlier allocations do not.

In [ ]:
class FIFO(Policy):
    def target_concurrency(self, view, state):
        return 4

    def choose_victim(self, view, candidates, state):
        return candidates[0]          # candidates arrive oldest-released first


h = BlockHarness(FIFO(), capacity=3)
h.allocate("A")
h.allocate("B")
show(h, "allocate A, B")

h.release("A")
h.release("B")
show(h, "release both")

h.allocate("C")
h.release("C")
show(h, "C takes the empty slot")

h.access("A")
h.release("A")
show(h, "hit on A, then A is released again")

h.allocate("D")
show(h, "D needs a slot: FIFO evicts the oldest release, B")

In the trace above, FIFO and LRU evict the same block (`B`) because `A` was both accessed and released most recently.

However, **release order (FIFO)** and **access order (LRU)** differ when a block is accessed more recently, but finishes and releases its reference *earlier* than an older block:

1. Call 1 allocates `A` (access order: `A`).
2. Call 2 allocates `B`, and Call 3 also hits `B` (access order: `A`, then `B`).
3. Calls 2 and 3 finish first, releasing `B`, while Call 1 finishes later, releasing `A` (release order: `B`, then `A`).

When the cache later needs space for `D`, which block will FIFO evict, and which block would LRU evict? Run the cell below to see FIFO's behavior.

In [ ]:
h = BlockHarness(FIFO(), capacity=3)
h.allocate("A")
h.allocate("B")
h.access("B")
show(h, "A held by one call, B by two")

h.release("B")
h.release("B")
h.release("A")
show(h, "B's calls finish first, then A's")

h.allocate("C")
h.allocate("D")
show(h, "FIFO evicts B, even though B was accessed more recently than A")

## 2. Implement LRU eviction

Now implement LRU eviction in `policies/reference.py` by completing `__init__`, `on_event`, and `choose_victim`:

- **`__init__`:** Initialize a data structure (such as ` collections.OrderedDict`) to track block IDs from least recently used to most recently used.
- **`on_event(self, event, state)`:**
  - When `event.kind in ("alloc", "hit")`, record `event.block` as the most recently used block (e.g., `move_to_end(event.block)`). Always follow event delivery order rather than comparing timestamps, since multiple events can share a timestamp.
  - When `event.kind == "evict"`, remove `event.block` from your recency tracker.
  - Ignore `"release"` and all other event kinds—releasing a block does not count as an access.
- **`choose_victim(self, view, candidates, state)`:**
  - Return the block ID in `candidates` that was least recently used according to your tracker. (Note: your tracker may also contain currently active blocks, which will not be in `candidates`.)

`policies/reference.py` also serves as the baseline starting policy for Part 3 (with default `cap = 8`). Run the unit tests and the 3-slot trace below to verify that your LRU policy evicts `A` instead of `B`.

In [ ]:
check("tests/test_part2_lru.py")

In [ ]:
h = BlockHarness(load_policy("policies/reference.py"), capacity=3)
h.allocate("A")
h.allocate("B")
h.access("B")

h.release("B")
h.release("B")
h.release("A")

h.allocate("C")
h.allocate("D")
show(h, "LRU evicts A: B was used more recently")

## 3. Evaluate LRU on agent traces

With LRU working, we can replay full workload traces in the Vidur simulator using `viz.record` and inspect how concurrency caps interact with KV-cache capacity.

We start with `dev-W3-0`, a **document QA** trace where users ask short questions about ~12,000-token documents from a shared library and the model generates short answers. In the two plots below (`cap = 8`):

- **Top plot (`viz.timeline`):** One row per job. **Yellow** = ready call waiting for a concurrency slot (`target_concurrency`); **Orange** = dispatched call waiting in the replica queue for engine admission; **Blue** = admitted call running prefill/decode; **Black tick (`|`)** = job arrival.
- **Bottom plot (`viz.memory`):** Stacked KV-cache block usage over time (**Blue** = active blocks held by running calls; **Green** = cached blocks available for prefix reuse; **Gray** = empty blocks).

In [ ]:
W3 = "data/traces/public_dev/dev-W3-0.json"
rec8 = viz.record(W3, "policies/reference.py", params={"cap": 8})

fig, axes = plt.subplots(2, 1, figsize=(12, 10), gridspec_kw={"height_ratios": [2.4, 1]})
viz.timeline(rec8, axes[0])
viz.memory(rec8, axes[1])
fig.tight_layout()
plt.show()

Look at the memory plot above: when 8 long-context calls run concurrently, active blocks (blue) consume almost the entire KV cache, leaving very little room for cached blocks (green) and forcing frequent evictions.

Next, replay `dev-W3-0` across `cap` values of `8, 4, 3, 2, 1`. Before running the cell, predict how lowering `cap` will trade batching concurrency against cache hit rate and prefill time.

In [ ]:
for cap in (8, 4, 3, 2, 1):
    r = viz.record(W3, "policies/reference.py", params={"cap": cap})
    mean_latency = sum(r.result.flows) / len(r.result.flows)
    hit = sum(c["cached"] for c in r.calls) / sum(c["prompt"] for c in r.calls)
    d = r.result.diag
    print(f"cap {cap}: mean job latency {mean_latency:5.2f} s   prompt tokens from cache {hit:.0%}   "
          f"engine busy {d['busy_time']:4.1f} s, prefilling {d['prefill_time']:4.1f} s")

Now run the same concurrency sweep on `dev-D-0`, a **deep research** trace with the same parallel-branch DAG structure as Part 1. Unlike document QA jobs, whose calls form one sequential chain, each research report fans out into parallel subtopic branches and generates longer extraction and synthesis outputs.

In [ ]:
D = "data/traces/public_dev/dev-D-0.json"
for cap in (8, 4, 2):
    r = viz.record(D, "policies/reference.py", params={"cap": cap})
    mean_latency = sum(r.result.flows) / len(r.result.flows)
    hit = sum(c["cached"] for c in r.calls) / sum(c["prompt"] for c in r.calls)
    d = r.result.diag
    print(f"cap {cap}: mean job latency {mean_latency:5.2f} s   prompt tokens from cache {hit:.0%}   "
          f"engine busy {d['busy_time']:4.1f} s, prefilling {d['prefill_time']:4.1f} s")

Compare the two workloads:

- **Document QA (`dev-W3-0`):** Every call has a ~12K-token prompt and a tiny output. The cache hit rate barely changes between caps 8 and 2, and the engine does about the same total work. The gain comes from scheduling: with a 2,048-token batch budget, eight concurrent long prefills share every batch and all finish late, while a lower cap lets a few calls finish early (compare time-sharing with running jobs one after another).
- **Deep research (`dev-D-0`):** Each report has parallel branches on its critical path and longer decode phases. Restricting `cap` too aggressively serializes parallel branches and increases slot-wait time. (Note: because Vidur batches prefill and decode together, `prefill_time` counts any batch step that includes prefill tokens.)

## 4. Concurrency caps on one and two replicas

Now evaluate LRU across all 32 development traces (`public_dev`, four per family) at several caps. Each grid scores the split four times and takes several minutes; the coding-agent traces (W2), with hundreds of thousands of evictions, take longest. `score()` computes each trace's speedup (reference mean job latency / candidate mean job latency), takes a geometric mean within each family, and then across families; the reference (`cap = 8`) scores 1.000 everywhere. The first grid uses one replica, the second the two replicas of Part 3, where each replica has the same memory as the single replica of the first.

![The eight workload families used for evaluation.](../docs/figures/families.svg)

In [ ]:
CAPS = (2, 4, 8, 16)


def cap_grid(replicas):
    # Score LRU at each cap and plot per-family speedups relative to cap 8.
    sweep = {cap: score("policies/reference.py", split="public_dev", replicas=replicas, params={"cap": cap})
             for cap in CAPS}
    bad = [r for r in sweep.values() if not r.valid]
    if bad:
        raise ValueError(f"policies/reference.py is invalid: {bad[0].violations[0]}")
    families = list(sweep[8].by_family)
    grid = [[sweep[c].by_family[f] for c in CAPS] for f in families] + [[sweep[c].speedup for c in CAPS]]
    viz.speedup_grid(grid, families + ["S"], CAPS, xlabel="cap",
                     title=f"LRU speedup over LRU at cap 8, {replicas} replica{'s' if replicas > 1 else ''}")
    plt.show()


cap_grid(replicas=1)

In [ ]:
cap_grid(replicas=2)

## 5. Prefix-aware eviction (Task 2)

Compare FIFO with LRU on one replica first:

In [ ]:
for path in ("policies/fifo.py", "policies/reference.py"):
    result = score(path, split="public_dev", replicas=1)
    print(f"{Path(path).name:13} S = {result.speedup:.3f}  " +
          " ".join(f"{f} {v:.2f}" for f, v in result.by_family.items()))

FIFO beats LRU on one replica because of how prefixes are matched:

![Evicting a root prevents reuse of its descendants; evicting a tail preserves the remaining prefix.](../docs/figures/evict_tail.svg)

1. A call touches its prompt's blocks from root to tail, so LRU considers the **root** of a prefix the least recently used.
2. Matching stops at the first missing block. Evicting the root strands every block after it: they stay in memory, but no prompt can reach them.
3. Vidur releases a finished call's blocks from tail to root, so FIFO, which evicts in release order, tends to evict **tails** and keep usable prefixes.

Production engines avoid the problem the same way: vLLM frees a finished request's blocks tail first, so its LRU free queue evicts tails before roots.

Implement `on_event` and `choose_victim` in `policies/prefix_policy.py`: evict the least recently used candidate that is a **leaf** of its replica's prefix tree (no resident block on the same replica extends it), and fall back to plain LRU when no candidate is a leaf. Keep the tree up to date from `alloc` and `evict` events (`alloc` carries the block's `prefix`, `parent`, and `replica`); do not scan `view.blocks`, which holds every resident block. Run the tests, then compare the policy with LRU on one and two replicas. `tails=False, slack=-1` makes the policy behave exactly like the reference.

In [ ]:
check("tests/test_part2_prefix.py")

In [ ]:
rows = {}
for replicas in (1, 2):
    for name, tails in (("LRU", False), ("prefix-aware", True)):
        params = {"tails": tails, "slack": -1}
        rows[replicas, name] = score("policies/prefix_policy.py", split="public_dev", replicas=replicas,
                                     params=params)
for (replicas, name), r in rows.items():
    print(f"{replicas} replica(s), {name:12} S = {r.speedup:.3f}  " +
          " ".join(f"{f} {v:.2f}" for f, v in r.by_family.items()))

## 6. Cache-affinity routing (Task 3)

With two replicas, a prefix cached on one replica is useless to a call sent to the other. The reference routes every call to the replica with fewer calls in flight, which balances load but scatters prefixes. Routing every call to the replica holding the longest cached prefix of its prompt does the opposite: when many calls share one system prompt, they all pile onto one replica.

Implement `route_call` in `policies/prefix_policy.py`: among replicas whose `in_flight` is at most `slack` above the least loaded replica's, choose the one with the longest cached prefix (`view.cached_prefix_len(call, replica)`), breaking ties by load and then replica id. `slack = -1` disables affinity. Then sweep `slack` with prefix-aware eviction enabled.

In [ ]:
SLACKS = (-1, 0, 1, 2, 4, 16)
routing = {s: score("policies/prefix_policy.py", split="public_dev", params={"slack": s}) for s in SLACKS}
families = list(routing[-1].by_family)
grid = [[routing[s].by_family[f] for s in SLACKS] for f in families] + [[routing[s].speedup for s in SLACKS]]
viz.speedup_grid(grid, families + ["S"], [str(s) for s in SLACKS], xlabel="slack (-1: least loaded)",
                 title="Prefix-aware eviction with cache-affinity routing, 2 replicas")
plt.show()

To see what a large slack does, replay a customer-service trace with `slack = 1` and `slack = 16`. Every call there shares one long policy prompt.

In [ ]:
W4 = "data/traces/public_dev/dev-W4-0.json"
fig, axes = plt.subplots(2, 1, figsize=(12, 11), layout="constrained")
for ax, slack in zip(axes, (1, 16)):
    rec = viz.record(W4, "policies/prefix_policy.py", params={"slack": slack}, replicas=2)
    mean = sum(rec.result.flows) / len(rec.result.flows)
    viz.timeline(rec, ax, title=f"slack = {slack}: mean job latency {mean:.2f} s")
plt.show()

## 7. Optional: job-level dispatch order

`dispatch_order(view, ready, state)` chooses which ready calls enter the engine next. Consider two 3-call jobs that arrive together, each call taking 1 s, with `cap = 1`. Interleaving them (`A1, B1, A2, B2, A3, B3`) finishes the jobs at 5 s and 6 s (mean 5.5 s); running the oldest job first (`A1, A2, A3, B1, B2, B3`) finishes them at 3 s and 6 s (mean 4.5 s) with the same work. To try it, copy `policies/prefix_policy.py` to `candidates/job_first.py`, add the hook below, and score both files.

```python
    def dispatch_order(self, view, ready, state):
        return sorted(ready, key=lambda c: view.jobs[view.ready[c].job].arrival)
```

## Questions

Answer Q1–Q3 below, keeping each question above its answer, and include them in `assignment4_report.pdf` after Part 1.

**Q1. Concurrency on one and two replicas.** Using the two cap grids, which families prefer a low cap on one replica, and why (refer to memory per call, prefix reuse, and the structure of their jobs)? Why does the best cap change with two replicas? Name one family where you would expect a single fixed cap to be wrong for part of the trace.

**Your answer:**

**Q2. Eviction and prefix trees.** Report LRU and prefix-aware eviction on one and two replicas. Which families gain most, and what about their prompts explains it? Why is the gain smaller with two replicas? Name a workload for which evicting tails first would be the wrong choice.

**Your answer:**

**Q3. Affinity versus load.** Plot or tabulate S and the per-family speedups across `slack`. Explain the best slack and the failure at large slack, using the two W4 timelines as evidence. What would you change in `route_call` to make large slack safe?

**Your answer:**